*Title:* 5_Manhattan Plots
*Goal:* Prepare data for manhatten plots and the graph.

# 5.1 Prepare data (temporal and spatio-temporal)
*Goal:* Prepare the file for plotting (add cutoff values, looping for 2016 and 2017 files)
Based off of data-manipulation-fst-output-file

``` R
module load r/4.4.0 
Rscript 3.e.ii.add_cutoff.R
``` 

RScript:
``` R
library(tidyverse)
library(data.table)

process_file <- function(input_file_name, output_file_name) {
  # Read the input file
  df <- read.table(input_file_name, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
  
  # Reshape the data from wide to long format
  tidy_data <- df %>%
    pivot_longer(cols = 2:7, names_to = "population", values_to = "Fst_poolfstat") %>%
    separate(Chromosome_Position, into = c("Chromosome", "Position"), sep = "_") %>% # splitting Chromosome and Position columns
    mutate(Chromosome = recode(Chromosome,
      "chrI" = 1, "chrII" = 2, "chrIII" = 3, "chrIV" = 4, "chrV" = 5,
      "chrVI" = 6, "chrVII" = 7, "chrVIII" = 8, "chrIX" = 9, "chrX" = 10,
      "chrXI" = 11, "chrXII" = 12, "chrXIII" = 13, "chrXIV" = 14, "chrXV" = 15,
      "chrXVI" = 16, "chrXVII" = 17, "chrXVIII" = 18, "chrXIX" = 19, "chrXX" = 20,
      "chrXXI" = 21
    )) %>% # converting Chromosome values to digits
    mutate(Cutoff = NA) # initialize Cutoff column with NA

  # Calculate outliers
  calculate_outliers <- function(tidy_data) {
    result_list <- list()
    unique_samples <- unique(tidy_data$population)
    
    for (sample in unique_samples) {
      subset_data <- tidy_data[tidy_data$population == sample, ]
      sub_Fst_poolfstat <- subset_data$Fst_poolfstat
      threshold <- quantile(sub_Fst_poolfstat, probs = 0.95, na.rm = TRUE) # 95th percentile threshold
      outliers <- which(sub_Fst_poolfstat > threshold)  # identifies outliers
      min_outlier <- min(sub_Fst_poolfstat[outliers], na.rm = TRUE)
      max_outlier <- max(sub_Fst_poolfstat[outliers], na.rm = TRUE)
      
      result_list[[sample]] <- list(CutoffValue = threshold, MinOutlier = min_outlier, MaxOutlier = max_outlier)
    }
    return(result_list)
  }
  
  results <- calculate_outliers(tidy_data)
  print(results)  # Print out the results for debugging

  # Add cutoff values to the data
  for (pop_name in names(results)) {
    match_rows <- tidy_data$population == pop_name
    tidy_data$Cutoff[match_rows] <- results[[pop_name]]$CutoffValue
  }

  # Write the processed data to the output file
  write.table(tidy_data, output_file_name, sep = "\t", col.names = TRUE, row.names = FALSE, quote = FALSE)
}

# Define input and output file names
#Making a loop for both 2016 abd 2017 to run at same time :)
input_file_1 <- "3.d.ii2017_poolfstat_SNP_FST_with_locations.tsv"
output_file_1 <- "3.e.ii2017_poolfstat_SNP_FST_with_locations_cutoff.tsv"

input_file_2 <- "3.d.ii2016_poolfstat_SNP_FST_with_locations.tsv"  
output_file_2 <- "3.e.ii2016_poolfstat_SNP_FST_with_locations_cutoff.tsv"  

# Process both files
process_file(input_file_1, output_file_1)
process_file(input_file_2, output_file_2)
``` 

# 5.2 Prepare unique population files for spatio-temporal
*Goal:* Prepare the data to generate plots for temporal and spatio-temporal outliers. 
Based off Generate_manhatten_plot


Manhatten Plots showing different types of repeatability: -------------------------------------
*Goal*: Split the file so each population gets its own output file . f


``` bash
#for 2017: (2016 version same as above but with 3.e.ii2016_poolfstat_SNP_FST_with_locations_cutoff.tsv instead!)

awk -F'\t' '
NR == 1 {
    header = $0
    for (i = 1; i <= NF; i++) {
        colname[i] = $i
    }
    next
}
{
    population = $3  # population column is the third column
    file = "FST_with_locations_cutoff_" population ".tsv"
    if (!(file in files)) {
        print header > file
        files[file]
    }
    print $0 >> file
}
' /home/mary2018/scratch/MultiYear/analysis/masked/REDO/poolfstat-fst-version-ii/3.e.ii2017_poolfstat_SNP_FST_with_locations_cutoff.tsv
```
Deal with the repeatability file
*Goal:* 
To make the files match, I also need to split the chrI_pos into separate columns. I want to make two new columns which are the splitting of Column_number by _:. I then want to rename the ChrI to numbers for matching (and later graphing):. The repeatability_cleaned.tsv file contains the repeatability counts for each position

``` bash
awk -F'\t' '
BEGIN {
    OFS = "\t"
    # Define the mapping of chromosome names to numbers
    chr_map["chrI"] = 1
    chr_map["chrII"] = 2
    chr_map["chrIII"] = 3
    chr_map["chrIV"] = 4
    chr_map["chrV"] = 5
    chr_map["chrVI"] = 6
    chr_map["chrVII"] = 7
    chr_map["chrVIII"] = 8
    chr_map["chrIX"] = 9
    chr_map["chrX"] = 10
    chr_map["chrXI"] = 11
    chr_map["chrXII"] = 12
    chr_map["chrXIII"] = 13
    chr_map["chrXIV"] = 14
    chr_map["chrXV"] = 15
    chr_map["chrXVI"] = 16
    chr_map["chrXVII"] = 17
    chr_map["chrXVIII"] = 18
    chr_map["chrXIX"] = 19
    chr_map["chrXX"] = 20
    chr_map["chrXXI"] = 21
}
NR == 1 {
    header = $0 "\tChr\tPos"
    print header
    next
}
{
    split($1, parts, "_")  # Split the first column (chr_pos)
    tChr = chr_map[parts[1]]  # Map the chromosome name to its corresponding number
    $0 = $0 "\t" tChr "\t" parts[2]
    print
}
' /home/mary2018/scratch/MultiYear/analysis/masked/REDO/poolfstat-fst-version-ii/repeatability/repeatability_cleaned_AGAIN.tsv  > repeatability_cleaned_manhatten.tsv
```
Bring the files together!
*Goal:* 
Now I want to merge the local and spatial counts for each pop file, if the chr and pos columns match!
Note that because the repeatability_counts file only contains info on the outliers, there will be many lines that do not match. Those lines will get a 0 for the t-counts and spatial-temp counts (as non of those sites will even be outliers...)
The t-estuary file lists if the outlier is temporal for that estuary, while the spatial-temp file lists the number of times it is a temporal outlier
Submitted the below script for each population
``` bash
awk -F'\t' '
BEGIN {
    OFS = "\t"
}
NR == FNR {
    if (FNR == 1) {
        # Store the headers of columns 13 and 5
        header13 = $13
        header3 = $3
        next
    }
    # Read file2 into an array
    key = $20 "_" $21  # Getting the chr and pos
    data[key] = $13 "\t" $3  # I want the spatial-temp and the Local count (for the given estuary). Need to change these columns depending on the estuary
    next
}
FNR == 1 {
    # Process the header of file1
    print $0, header13, header3
    next
}
{
    # Process file1
    key = $1 "_" $2  # Chr and pos.
    if (key in data) {
        $0 = $0 "\t" data[key]
    } else {
        $0 = $0 "\tNA\tNA"  # Add NA if no match is found
    }
    print
}
' /home/mary2018/scratch/MultiYear/analysis/masked/REDO/poolfstat-fst-version-ii/repeatability/manhatten_repeat/repeatability_cleaned_manhatten.tsv FST_with_locations_cutoff_Younger2017.tsv  > FST_with_locations_cutoff_Younger2017_repeat.tsv
```
Tidy New Files:
*Goal:* 
Replacing NA with 0
Adjusting spatial-temp counts
Need to get the spatial_repeatability count, but only show if it is a local outlier for that estuary (currently a count of 2 means that in the dataset, it is temporally repeated twice, but that doesn't mean that it is a spatial-repeated outlier (2) for the given estuary). Instead now, a 2 means that it is a temp-spatial outlier in Estuary A and another estuary. My goal here is to show temp-spatial repeatability counts for each estuary, and only show outliers that are outliers in the estuary of interest.
``` R
library(data.table)
library(tidyverse)
files <- list.files(pattern = "_repeat.tsv$")
process_file <- function(file) {
data <- read.table(file, header = TRUE, sep = "\t")
file_base <- gsub("FST_with_locations_cutoff_|_repeat.tsv", "", file)
output_file <- paste0(file_base, "_modified.tsv")
new_col_name <- paste0("spatial_temp_", file_base) #adjusting the column name so it is different
data[is.na(data)] <- 0 #Replace all NA values with 0
data[, 6] <- ifelse(data[, 7] == 0, 0, data[, 6]) #adjusting the count for spatial-temp
colnames(data)[6] <- new_col_name
write.table(data, output_file, sep = "\t", row.names = FALSE, quote = FALSE)
}
lapply(files, process_file)
```

# 5.3 Prepare data for spatio-only plots
Based on scripts from FST_by_repeatability_type

*Goal:* Generate a file with the spatial_only column, and the FST value
The file: repeatability_cleaned_manhatten.tsv contains the counts of repeatability types of each SNP
The other file contains the FST values for the estuary of interest.

``` bash
#For 2016:
#Loop through all files matching the pattern FST_with_locations_cutoff_*.tsv
for file in FST_with_locations_cutoff_*6.tsv; do
    # Extract the base name of the file (without extension)
    base_name=$(basename "$file" .tsv)
    
    # Define the output file name
    output_file="${base_name}_spatial_only.tsv"
    

awk -F'\t' '
BEGIN {
    OFS = "\t"
}
NR == FNR {
    if (FNR == 1) {
        # Store the spatial_only columns 
        only_spatial = $11
        next
    }
    # Read file2 into an array
    key = $20 "_" $21  # Getting the chr and pos
    data[key] = $11
    next
}
FNR == 1 {
    # Process the header of file1
    print $0, only_spatial
    next
}
{
    # Process file1
    key = $1 "_" $2  # Chr and pos.
    if (key in data) {
        $0 = $0 "\t" data[key]
    } else {
        $0 = $0 "\tNA"  # Add NA if no match is found
    }
    print
}
  ' "$counts_file" "$file" > "$output_file"
  echo "Processed $file -> $output_file"
done

# The 2017 version:
counts_file="repeatability_cleaned_manhatten.tsv"

# Loop through all files matching the pattern FST_with_locations_cutoff_*.tsv
for file in FST_with_locations_cutoff_*7.tsv; do
    # Extract the base name of the file (without extension)
    base_name=$(basename "$file" .tsv)
    
    # Define the output file name
    output_file="${base_name}_spatial_only.tsv"
    

awk -F'\t' '
BEGIN {
    OFS = "\t"
}
NR == FNR {
    if (FNR == 1) {
        # Store the spatial_only columns 
        only_spatial = $12
        next
    }
    # Read file2 into an array
    key = $20 "_" $21  # Getting the chr and pos
    data[key] = $12
    next
}
FNR == 1 {
    # Process the header of file1
    print $0, only_spatial
    next
}
{
    # Process file1
    key = $1 "_" $2  # Chr and pos.
    if (key in data) {
        $0 = $0 "\t" data[key]
    } else {
        $0 = $0 "\tNA"  # Add NA if no match is found
    }
    print
}
  ' "$counts_file" "$file" > "$output_file"
  echo "Processed $file -> $output_file"
done
```

Tidy all files

*Goal:* I now want to turn all NA to 0
I also need to adjust the spatial_only counts
Currently, the counts are just psated for that site, meaning that it is the number of estuaries that are spatial outliers (only) at that SNP.
I need it to only show up if that SNP is an outlier for that estuary 
Meaning a 2 should be that it is only a spatial outlier in that estuary and another, in that hyear
Only keep the spatial_only count if the fst value is > than the cutoff (i.e. it is an outlier):

```R
library(data.table)
library(tidyverse)
files <- list.files(pattern = "_spatial_only.tsv$")
process_file <- function(file) {
data <- read.table(file, header = TRUE, sep = "\t")
base_name <- tools::file_path_sans_ext(basename(file))
output_file <- paste0(base_name, "_cleaned.tsv")
any(is.na(data[, 1:5]))#Check if NA in any of hte other columns (should be false)
data[is.na(data)] <- 0 #Replace all NA values with 0
#new_col_name <- spatial_only
data[, 6] <- ifelse(data[, 4] > data[, 5], data[, 6],0) #adjusting the only count so that if the fst > cutoff (i.e. it is an outlier), the spatial_only count remains. If it isn't (i.e. not an outlier), it turns into 0.
write.table(data, output_file, sep = "\t", row.names = FALSE, quote = FALSE)
}
lapply(files, process_file)
```


# 5.4 Spatio-Temporal Manhatten Plots:
*Goal:* Generate manhatten plots (multipanel) showing temporal and spatio-temporally repeated outliers.
Based off Generate_manhatten_plot

Manhatten Plots of spatial temporal unique MULTIPANEL -------------------------------------
```R
library(cowplot)
library(tidyverse)
library(ggplot2)

files <- list.files(pattern = "_modified.tsv$")
process_file <- function(file) {
    file_base <- basename(file)
    print(paste("Processing file:", file_base))
  data <- read.table(file, header = TRUE, sep = "\t")
  
  data$Position <- as.integer(data$Position)
  data$Chromosome <- as.integer(data$Chromosome)
  data$Fst_poolfstat <- as.numeric(data$Fst_poolfstat)
  data$Cutoff <- as.numeric(data$Cutoff)
  data[, 6] <- as.factor(data[, 6]) # the spatial_temp_for the estuary needs to be a factor
  colnames(data)[6] <- "spa_temp"
  str(data)
  
  create_population_manhplot <- function(data) {
    data_cum <- data %>%
      group_by(Chromosome) %>%
      summarise(max_bp = max(Position)) %>%
      mutate(bp_add = lag(cumsum(max_bp), default = 0)) %>%
      select(Chromosome, bp_add)
    
    data <- data %>%
      inner_join(data_cum, by = "Chromosome") %>%
      mutate(bp_cum = Position + bp_add)
    
    axis_set <- data %>%
      group_by(Chromosome) %>%
      summarise(center = mean(bp_cum))
    
    manhplot <- ggplot(data, aes(x = bp_cum, y = Fst_poolfstat, color = factor(spa_temp), size = ifelse(spa_temp == "0", 0.05, 0.3), alpha = ifelse(spa_temp == "0", 0.2, 0.8))) +
      geom_point() +
      scale_color_manual(values = c("1" = "#7FBDFF", "2" = "#00859E", "3" = "#AF0072", '0' = '#AFADAF')) +
      scale_size_identity() +
      scale_alpha_identity() +
      scale_x_continuous(
        labels = axis_set$Chromosome,
        breaks = axis_set$center
      ) +
      geom_hline(aes(yintercept = Cutoff, linetype = "Cutoff"), color = "grey3", show.legend = FALSE) + 
      scale_linetype_manual(name = "Cutoff Value", values = "dashed", labels = paste("5%= ", sprintf("%.3f", unique(data$Cutoff)))) +
      scale_y_continuous(limits = c(0, 1)) +
      labs(x = "Genomic Position", y = "FST Value") +
      theme_minimal() +
       annotate("text", x = Inf, y = Inf, label = paste("5% = ", sprintf("%.3f", unique(data$Cutoff))), hjust = 1.1, vjust = 1.5, size = 3, color = "black") + #adding text to say 5% NEW ADDITION JUNE 12!
      theme(
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank(),
        axis.title.y = element_text(),
        axis.text.x = element_text(angle = 60, size = 8, vjust = 0.5),
        plot.background = element_rect(fill = "white"),
        panel.background = element_rect(fill = "white"),
        legend.position = "none",
        axis.title.x = element_blank(), #removing axis title
      )
    
    return(manhplot) 
  }
  
  plot <- create_population_manhplot(data)
  return(plot)
}

plots <- lapply(files, process_file)

plots_with_margins <- lapply(plots, function(p) {
  p + theme(plot.margin = margin(t = 20, b = 5.5, l=5.5, r=5.5)) # Increase top and bottom margins
})

combined_plot <- plot_grid(
  plots_with_margins[[1]], plots_with_margins[[2]], plots_with_margins[[3]], plots_with_margins[[4]], plots_with_margins[[5]], plots_with_margins[[6]], 
  plots_with_margins[[7]], plots_with_margins[[8]], plots_with_margins[[9]], plots_with_margins[[10]], plots_with_margins[[11]], plots_with_margins[[12]], 
  ncol = 2, nrow = 6,
  rel_heights = c(1, 1, 1, 1, 1, 1),
  labels = c(
    "Laguna 2016", "Laguna 2017", "Lombardi 2016", "Lombardi 2017", 
    "Old Dairy 2016", "Old Dairy 2017", "Scott 2016", "Scott 2017", 
    "Waddell 2016", "Waddell 2017", "Younger 2016", "Younger 2017"
  ), 
  label_size = 12
)

legend_st <- data.frame(
  x = 1:4,
  y = runif(4, 0, 1),
  group = factor(c(0,1,2,3)))

legend_st_plot <- ggplot(legend_st, aes(x = x, y = y, color = group)) +
  geom_point(size = 3) +
  scale_color_manual(
    values = c('0' = '#AFADAF',"1" = "#7FBDFF", "2" = "#00859E", "3" = "#AF0072"),
    labels = c("Non-spatio-temporal","Temporal","Spatio-temporal 2","Spatio-temporal 3")
  ) +
  labs(color = "Repeatability Type") +
  theme_minimal() +
  theme(legend.position="bottom",
        legend.spacing.x = unit(0, 'cm'),
        legend.title = element_text(size = 14, face = "bold"),
    legend.text = element_text(size = 12),
    )+
  guides(fill = guide_legend(label.position = "bottom"))

#Extract the Legend
legend_grob <- cowplot::get_legend(legend_st_plot)
legend_plot <- ggdraw() + draw_grob(legend_grob) +
  theme(plot.background = element_rect(fill = "white", colour = NA))

# Now combine the combined and legend:
combined_plot_legend <- plot_grid(combined_plot,
legend_plot ,
   ncol = 1, 
   nrow=2,
rel_heights = c(1, 0.10),
  label_size = 12,
  align='v'
)
ggsave("combined_plot_legend_st.png", combined_plot_legend, width = 11, height = 16, units = "in", dpi = 1000)
ggsave("combined_plot_legend_st.pdf", combined_plot_legend, width = 11, height = 16, units = "in", dpi = 1000)
```
Manhatten Plots Spatial -------------------------------------
```R
library(ggplot2)
library(dplyr)
library(data.table)
library(tidyverse)
library(cowplot)

 files <- list.files(pattern = "_spatial_only_cleaned\\.tsv$")
process_file <- function(file, idx= NULL) {
    file_base <- basename(file)
  print(paste("Processing file:", file_base))
data <- read.table(file, header = TRUE, sep = "\t")
  data$Position <- as.integer(data$Position)
  data$Chromosome <- as.integer(data$Chromosome)
  data$Fst_poolfstat <- as.numeric(data$Fst_poolfstat)
  data$Cutoff <- as.numeric(data$Cutoff)
  data[, 6] <- as.factor(data[, 6]) # the spatial_temp_for the estuary needs to be a factor
  colnames(data)[6] <- "spatial_only"
  print(table(data$spatial_only)) 
  str(data)
  
  create_population_manhplot <- function(data) {
    data_cum <- data %>%
      group_by(Chromosome) %>%
      summarise(max_bp = max(Position)) %>%
      mutate(bp_add = lag(cumsum(max_bp), default = 0)) %>%
      select(Chromosome, bp_add)
    
    data <- data %>%
      inner_join(data_cum, by = "Chromosome") %>%
      mutate(bp_cum = Position + bp_add)
    
    axis_set <- data %>%
      group_by(Chromosome) %>%
      summarise(center = mean(bp_cum))

    
    manhplot <- ggplot(data, aes(x = bp_cum, y = Fst_poolfstat, color = factor(spatial_only), size = ifelse(spatial_only %in% c("0", "1"), 0.05, 0.3), alpha = ifelse(spatial_only %in% c("0", "1"), 0.2, 0.8))) + #counts of 0 and 1 have the same alpha and size 
      geom_point() +
      scale_color_manual(values = c("1" = "#AFADAF", "2" = "#EF6604", "3" = "#E31A1C", '4' = '#B10026', '5' = '#A200B1', '6' = '#0041B1', '0' = '#AFADAF')) + #colour spatial 1 and 0 the same because im not interest in spatial 1
      scale_size_identity() +
      scale_alpha_identity() +
      scale_x_continuous(
        labels = axis_set$Chromosome,
        breaks = axis_set$center
      ) +
      geom_hline(aes(yintercept = Cutoff, linetype = "Cutoff"), color = "grey3", show.legend = FALSE) + 
      scale_linetype_manual(name = "Cutoff Value", values = "dashed", labels = paste("5%= ", sprintf("%.3f", unique(data$Cutoff)))) +
      scale_y_continuous(limits = c(0, 1)) +
      labs(y = "FST Value") +
      annotate("text", x = Inf, y = Inf, label = paste("5% = ", sprintf("%.3f", unique(data$Cutoff))), hjust = 1.1, vjust = 1.5, size = 3, color = "black") + #adding text to say 5%
      theme_minimal() +
      theme(
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank(),
        axis.title.y = element_text(),
        axis.title.x = element_blank(), 
        axis.text.x = element_text(angle = 60, size = 8, vjust = 0.5, hjust=1),
        plot.background = element_rect(fill = "white"),
        panel.background = element_rect(fill = "white"),
        legend.position = "none"
      )
    
    return(manhplot) 
  }


  plot <- create_population_manhplot(data)
  return(plot)
}

plots <- lapply(files, process_file)
plots_with_margins <- lapply(plots, function(p) {
  p + theme(plot.margin = margin(t = 20, b = 5.5, l=5.5, r=5.5)) # Increase top and bottom margins
})

combined_plot <- plot_grid(
  plots_with_margins[[1]], plots_with_margins[[2]], plots_with_margins[[3]], plots_with_margins[[4]], plots_with_margins[[5]], plots_with_margins[[6]], 
  plots_with_margins[[7]], plots_with_margins[[8]], plots_with_margins[[9]], plots_with_margins[[10]], plots_with_margins[[11]], plots_with_margins[[12]], 
  ncol = 2, nrow = 6,
  rel_heights = c(1, 1, 1, 1, 1, 1),
  labels = c(
    "Laguna 2016", "Laguna 2017", "Lombardi 2016", "Lombardi 2017", 
    "Old Dairy 2016", "Old Dairy 2017", "Scott 2016", "Scott 2017", 
    "Waddell 2016", "Waddell 2017", "Younger 2016", "Younger 2017"
  ), 
  label_size = 12
)

# Getting a legend:
#making the legend horizontal and at the bottom
legend_spatial <- data.frame(
  x = 1:6,
  y = runif(6, 0, 1),
  group = factor(c(0,2,3,4,5,6)))

legend_spatial_plot <- ggplot(legend_spatial, aes(x = x, y = y, color = group)) +
  geom_point(size = 3) +
  scale_color_manual(
    values = c('0' = '#AFADAF',"2" = "#EF6604", "3" = "#E31A1C",'4' = '#B10026', '5' = '#A200B1', '6' = '#0041B1'),
    labels = c("Non-spatial","Spatial 2","Spatial 3","Spatial 4","Spatial 5","Spatial 6")
  ) +
  labs(color = "Repeatability Type") +
  theme_minimal() +
  theme(legend.position="bottom",
        legend.spacing.x = unit(0, 'cm'),
        legend.title = element_text(size = 14, face = "bold"),
    legend.text = element_text(size = 12),
    )+
  guides(fill = guide_legend(label.position = "bottom"))

#Extract the Legend
legend_grob <- cowplot::get_legend(legend_spatial_plot)
legend_plot <- ggdraw() + draw_grob(legend_grob) +
  theme(plot.background = element_rect(fill = "white", colour = NA))

# Now combine the Spatial and legend:
combined_plot_legend <- plot_grid(combined_plot,
legend_plot ,
   ncol = 1, 
   nrow=2,
rel_heights = c(1, 0.10),
  label_size = 12,
  align='v'
)
ggsave("combined_plot_legend_spatial.png", combined_plot_legend, width = 11, height = 16, units = "in", dpi = 1000)
ggsave("combined_plot_legend_spatial.pdf", combined_plot_legend, width = 11, height = 16, units = "in", dpi = 1000)
```